# 병원 슬라이드 데이터 Fine-Tuning

## 사전 준비 (Colab 실행 전 로컬에서)
1. `python app.py` 실행 후 병원 슬라이드 이미지 몇 장 분석
2. `PB_Smear/static/crops/` 폴더에 crop 이미지들이 저장됨
3. 그 이미지들을 직접 보면서 아래 구조로 분류:
```
hospital_crops/
  NEUTROPHIL/   ← 분엽핵, 가장 많음, 20장 이상
  MONOCYTE/     ← 콩팥형 핵, 10장 이상
  LYMPHOCYTE/   ← 작고 핵이 꽉 참, 10장 이상
  EOSINOPHIL/   ← 주황빛 과립, 있으면 5장 이상
  BASOPHIL/     ← 거의 없어도 됨
```
4. `hospital_crops/` 폴더를 구글 드라이브의 `PB_Smear/` 안에 업로드
5. 기존 `model.h5`도 드라이브의 `PB_Smear/` 안에 있어야 함

## 실행 순서
셀을 위에서 아래로 순서대로 실행

In [ ]:
# =============================================
# [Step 1] 구글 드라이브 마운트
# =============================================
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR        = '/content/drive/MyDrive/PB_Smear'
HOSPITAL_CROPS   = f'{DRIVE_DIR}/hospital_crops'
MODEL_PATH       = f'{DRIVE_DIR}/model.h5'
OUTPUT_MODEL     = f'{DRIVE_DIR}/model_finetuned.h5'

import os
assert os.path.exists(HOSPITAL_CROPS), f'hospital_crops 폴더가 없습니다: {HOSPITAL_CROPS}'
assert os.path.exists(MODEL_PATH),     f'model.h5가 없습니다: {MODEL_PATH}'

# 클래스별 이미지 수 확인
for cls in os.listdir(HOSPITAL_CROPS):
    p = os.path.join(HOSPITAL_CROPS, cls)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        print(f'  {cls}: {n}장')

In [ ]:
# =============================================
# [Step 2] 패키지 설치
# =============================================
!pip install tensorflow -q
import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# =============================================
# [Step 3] 설정
# =============================================
IMG_SIZE   = 224
BATCH_SIZE = 8      # 병원 crop 수가 적으므로 작게
EPOCHS_FT  = 20     # fine-tuning 에포크
LR_FT      = 2e-5   # 아주 낮은 학습률 (기존 가중치 보존)

# 원래 학습 때와 동일한 클래스 순서 (알파벳)
CLASS_NAMES = ['BASOPHIL', 'EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']

In [ ]:
# =============================================
# [Step 4] 데이터 제너레이터 (강력한 Augmentation)
# 병원 crop이 적으므로 augmentation을 극대화해 과적합 방지
# =============================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

ft_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=360,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='reflect',
)

train_gen = ft_datagen.flow_from_directory(
    HOSPITAL_CROPS,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
)
val_gen = ft_datagen.flow_from_directory(
    HOSPITAL_CROPS,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
)

print('클래스 인덱스:', train_gen.class_indices)
print(f'학습: {train_gen.samples}장 / 검증: {val_gen.samples}장')

In [ ]:
# =============================================
# [Step 5] 기존 model.h5 로드 + 상위 레이어 unfreeze
# =============================================
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

# EfficientNetB0 상위 50개 레이어만 unfreeze (하위는 고정)
# 병원 데이터가 적으므로 너무 많이 풀면 기존 학습 지식이 손상됨
for layer in model.layers:
    layer.trainable = False
for layer in model.layers[-50:]:
    layer.trainable = True

trainable = sum(1 for l in model.layers if l.trainable)
print(f'학습 가능한 레이어: {trainable}개')

# 클래스 가중치 계산
labels = train_gen.classes
unique_cls = np.unique(labels)
weights = compute_class_weight('balanced', classes=unique_cls, y=labels)
class_weight = {int(c): min(float(w), 5.0) for c, w in zip(unique_cls, weights)}
print('클래스 가중치:', class_weight)

In [ ]:
# =============================================
# [Step 6] 컴파일 + Fine-Tuning
# =============================================
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

model.compile(
    optimizer=Adam(learning_rate=LR_FT),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(OUTPUT_MODEL, monitor='val_accuracy', save_best_only=True, verbose=1),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FT,
    class_weight=class_weight,
    callbacks=callbacks,
)

print('Fine-tuning 완료!')

In [ ]:
# =============================================
# [Step 7] 결과 확인 + 저장 안내
# =============================================
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],     label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend(); plt.title('Accuracy')
plt.subplot(1,2,2)
plt.plot(history.history['loss'],     label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend(); plt.title('Loss')
plt.tight_layout(); plt.show()

print(f'\n저장 완료: {OUTPUT_MODEL}')
print()
print('다음 단계:')
print('  1. 구글 드라이브 > PB_Smear > model_finetuned.h5 다운로드')
print('  2. 맥북 PB_Smear/ 폴더에 model_finetuned.h5 넣기')
print('  3. app.py에서 CLASSIFIER_PATH = "model_finetuned.h5" 로 변경')
print('  4. python app.py 재실행')